# SQL analytics layer

### PostgreSQL data model and business reporting views

This notebook loads the final out-of-sample forecasting and procurement
backtest into PostgreSQL.

The objective is to build a simple relational analytics layer on top of
the forecasting pipeline and answer business questions using SQL rather
than pandas.

The database contains hourly demand forecasts, market prices and
strategy-level procurement results. Analytical views provide
normal-operation strategy summaries, monthly procurement costs and
economic savings for downstream reporting.

In [1]:
from pathlib import Path
import os

import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from energy_forecasting.procurement import (
    build_procurement_results,
)

## 1. Load the final backtest artifact

The hourly backtest produced in the procurement notebook is the single
source of truth for the SQL analytics layer.

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = (
    project_root
    / "data"
    / "processed"
)

backtest = pd.read_csv(
    processed_dir / "costs.csv",
    index_col="datetime_utc",
    parse_dates=["datetime_utc"],
)

backtest.index = pd.to_datetime(
    backtest.index,
    utc=True,
)

backtest["datetime_local"] = pd.to_datetime(
    backtest["datetime_local"],
    utc=True,
).dt.tz_convert("Europe/Madrid")

In [3]:
assert backtest.index.is_unique
assert backtest.index.is_monotonic_increasing
assert backtest.isna().sum().sum() == 0

assert len(backtest) == 14591
assert backtest["is_blackout"].sum() == 19

print("Rows:", len(backtest))
print("Start:", backtest.index.min())
print("End:", backtest.index.max())

backtest.head()

Rows: 14591
Start: 2024-12-31 23:00:00+00:00
End: 2026-08-31 21:00:00+00:00


,datetime_local,demand_MW,forecast_ml,is_blackout,forecast_ree,forecast_lag168,day_ahead_price,imbalance_price_up,imbalance_price_down
datetime_utc,,,,,,,,,
2024-12-31 23:00:00+00:00,2025-01-01 00:00:00+01:00,22924.000000,20971.785247,False,23530.8,21646.500000,134.49,112.2525,133.3750
2025-01-01 00:00:00+00:00,2025-01-01 01:00:00+01:00,22185.916667,21096.561097,False,22670.3,20737.250000,131.59,114.5025,114.5025
2025-01-01 01:00:00+00:00,2025-01-01 02:00:00+01:00,20952.833333,19989.772718,False,21420.5,19471.416667,131.49,93.8950,140.2250
2025-01-01 02:00:00+00:00,2025-01-01 03:00:00+01:00,19774.750000,19386.697825,False,20285.0,18462.083333,131.42,118.2625,118.2350
2025-01-01 03:00:00+00:00,2025-01-01 04:00:00+01:00,18973.583333,19084.245030,False,19470.5,17824.333333,120.49,86.7200,118.6775


## 2. Relational data model

The hourly backtest is separated into three logical tables:

- `demand_forecasts`: realized demand and the three competing forecasts.
- `market_prices`: day-ahead and imbalance-market prices.
- `procurement_results`: strategy-level forecast errors and procurement costs.

This avoids duplicating demand and price information for each forecasting
strategy while preserving a simple timestamp-based relational structure.

In [4]:
demand_forecasts = backtest[
    [
        "demand_MW",
        "forecast_lag168",
        "forecast_ml",
        "forecast_ree",
        "is_blackout",
    ]
].copy()

demand_forecasts = demand_forecasts.rename(
    columns={
        "demand_MW":
            "actual_demand_mw",

        "forecast_lag168":
            "lag168_forecast_mw",

        "forecast_ml":
            "ml_forecast_mw",

        "forecast_ree":
            "ree_forecast_mw",
    }
)

demand_forecasts.head()

,actual_demand_mw,lag168_forecast_mw,ml_forecast_mw,ree_forecast_mw,is_blackout
datetime_utc,,,,,
2024-12-31 23:00:00+00:00,22924.000000,21646.500000,20971.785247,23530.8,False
2025-01-01 00:00:00+00:00,22185.916667,20737.250000,21096.561097,22670.3,False
2025-01-01 01:00:00+00:00,20952.833333,19471.416667,19989.772718,21420.5,False
2025-01-01 02:00:00+00:00,19774.750000,18462.083333,19386.697825,20285.0,False
2025-01-01 03:00:00+00:00,18973.583333,17824.333333,19084.245030,19470.5,False


In [5]:
market_prices = backtest[
    [
        "day_ahead_price",
        "imbalance_price_up",
        "imbalance_price_down",
    ]
].copy()

market_prices = market_prices.rename(
    columns={
        "day_ahead_price":
            "day_ahead_eur_mwh",

        "imbalance_price_up":
            "imbalance_up_eur_mwh",

        "imbalance_price_down":
            "imbalance_down_eur_mwh",
    }
)

market_prices.head()

,day_ahead_eur_mwh,imbalance_up_eur_mwh,imbalance_down_eur_mwh
datetime_utc,,,
2024-12-31 23:00:00+00:00,134.49,112.2525,133.3750
2025-01-01 00:00:00+00:00,131.59,114.5025,114.5025
2025-01-01 01:00:00+00:00,131.49,93.8950,140.2250
2025-01-01 02:00:00+00:00,131.42,118.2625,118.2350
2025-01-01 03:00:00+00:00,120.49,86.7200,118.6775


In [6]:
strategies = {
    "lag168": "forecast_lag168",
    "ml": "forecast_ml",
    "ree": "forecast_ree",
}

procurement_results = pd.concat(
    [
        build_procurement_results(
            data=backtest,
            forecast_col=forecast_col,
            strategy=strategy,
        )
        for strategy, forecast_col
        in strategies.items()
    ]
)

procurement_results.index.name = (
    "timestamp_utc"
)

procurement_results.head()

,strategy,forecast_mw,forecast_error_mw,shortfall_mwh,surplus_mwh,day_ahead_cost_eur,imbalance_settlement_eur,total_cost_eur,incremental_cost_eur
timestamp_utc,,,,,,,,,
2024-12-31 23:00:00+00:00,lag168,21646.500000,1277.500000,1277.500000,0.0,2.911238e+06,170386.562500,3.081624e+06,-1424.412500
2025-01-01 00:00:00+00:00,lag168,20737.250000,1448.666667,1448.666667,0.0,2.728815e+06,165875.955000,2.894691e+06,-24754.091667
2025-01-01 01:00:00+00:00,lag168,19471.416667,1481.416667,1481.416667,0.0,2.560297e+06,207731.652083,2.768028e+06,12940.174583
2025-01-01 02:00:00+00:00,lag168,18462.083333,1312.666667,1312.666667,0.0,2.426287e+06,155203.143333,2.581490e+06,-17307.510000
2025-01-01 03:00:00+00:00,lag168,17824.333333,1149.250000,1149.250000,0.0,2.147654e+06,136390.116875,2.284044e+06,-2083.015625


In [7]:
assert len(demand_forecasts) == 14591
assert len(market_prices) == 14591

assert len(procurement_results) == (
    3 * len(backtest)
)

assert (
    demand_forecasts
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    market_prices
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    procurement_results
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    demand_forecasts.index.is_unique
)

assert (
    market_prices.index.is_unique
)

print(
    "demand_forecasts:",
    len(demand_forecasts),
)

print(
    "market_prices:",
    len(market_prices),
)

print(
    "procurement_results:",
    len(procurement_results),
)

demand_forecasts: 14591
market_prices: 14591
procurement_results: 43773


In [8]:
demand_forecasts.index.name = "timestamp_utc"
market_prices.index.name = "timestamp_utc"
procurement_results.index.name = "timestamp_utc"

demand_forecasts_sql = (
    demand_forecasts
    .reset_index()
)

market_prices_sql = (
    market_prices
    .reset_index()
)

procurement_results_sql = (
    procurement_results
    .reset_index()
)

In [9]:
assert not (
    procurement_results_sql[
        [
            "timestamp_utc",
            "strategy",
        ]
    ]
    .duplicated()
    .any()
)

## 3. PostgreSQL connection

The local PostgreSQL database is accessed through SQLAlchemy.

The default connection assumes a local database named
`energy_analytics`. A custom connection string can be supplied through
the `ENERGY_DB_URL` environment variable without exposing credentials
in the repository.

In [10]:
load_dotenv(
    project_root / ".env"
)

database_url = os.getenv(
    "ENERGY_DB_URL",
    (
        "postgresql+psycopg://"
        "localhost:5432/"
        "energy_analytics"
    ),
)

engine = create_engine(
    database_url
)

with engine.connect() as conn:
    conn.execute(
        text("SELECT 1")
    )

print("PostgreSQL connection successful.")

PostgreSQL connection successful.


## 4. Load the base tables

The tables are replaced on each notebook run so that PostgreSQL always
reflects the latest backtest artifacts rather than accumulating stale
rows from previous model versions.

In [11]:
reset_sql = """
DROP VIEW IF EXISTS vw_monthly_savings CASCADE;
DROP VIEW IF EXISTS vw_monthly_costs CASCADE;
DROP VIEW IF EXISTS vw_strategy_summary CASCADE;
DROP VIEW IF EXISTS vw_hourly_operations CASCADE;

DROP TABLE IF EXISTS procurement_results CASCADE;
DROP TABLE IF EXISTS market_prices CASCADE;
DROP TABLE IF EXISTS demand_forecasts CASCADE;
"""

with engine.begin() as conn:
    conn.execute(text(reset_sql))

In [12]:
demand_forecasts_sql.to_sql(
    "demand_forecasts",
    engine,
    if_exists="fail",
    index=False,
)

market_prices_sql.to_sql(
    "market_prices",
    engine,
    if_exists="fail",
    index=False,
)

procurement_results_sql.to_sql(
    "procurement_results",
    engine,
    if_exists="fail",
    index=False,
)

-1

In [13]:
index_sql = """
CREATE UNIQUE INDEX IF NOT EXISTS
    idx_demand_forecasts_timestamp
ON demand_forecasts(timestamp_utc);

CREATE UNIQUE INDEX IF NOT EXISTS
    idx_market_prices_timestamp
ON market_prices(timestamp_utc);

CREATE UNIQUE INDEX IF NOT EXISTS
    idx_procurement_timestamp_strategy
ON procurement_results(
    timestamp_utc,
    strategy
);
"""

with engine.begin() as conn:
    for statement in (
        index_sql
        .strip()
        .split(";")
    ):
        if statement.strip():
            conn.execute(
                text(statement)
            )

In [14]:
table_counts = pd.read_sql(
    """
    SELECT
        'demand_forecasts' AS table_name,
        COUNT(*) AS n_rows
    FROM demand_forecasts

    UNION ALL

    SELECT
        'market_prices',
        COUNT(*)
    FROM market_prices

    UNION ALL

    SELECT
        'procurement_results',
        COUNT(*)
    FROM procurement_results;
    """,
    engine,
)

table_counts

,table_name,n_rows
0,demand_forecasts,14591
1,market_prices,14591
2,procurement_results,43773


In [15]:
counts = (
    table_counts
    .set_index("table_name")
    ["n_rows"]
    .to_dict()
)

assert counts[
    "demand_forecasts"
] == 14591

assert counts[
    "market_prices"
] == 14591

assert counts[
    "procurement_results"
] == 43773

## 5. Analytical SQL views

Four views provide the reporting layer:

`vw_strategy_summary` summarizes normal-operation procurement
performance by forecasting strategy.

`vw_monthly_costs` measures strategy costs by local Spanish calendar
month.

`vw_monthly_savings` converts those costs into relative economic
savings.

`vw_hourly_operations` exposes the joined hourly operational dataset
for detailed reporting.

In [16]:
strategy_summary_sql = """
CREATE OR REPLACE VIEW
vw_strategy_summary AS

SELECT
    pr.strategy,

    COUNT(*) AS n_hours,

    SUM(
        pr.incremental_cost_eur
    ) AS incremental_cost_eur,

    SUM(
        df.actual_demand_mw
    ) AS supplied_mwh,

    SUM(
        pr.incremental_cost_eur
    )
    /
    SUM(
        df.actual_demand_mw
    ) AS incremental_cost_eur_per_mwh

FROM procurement_results pr

JOIN demand_forecasts df
    ON pr.timestamp_utc
     = df.timestamp_utc

WHERE df.is_blackout = FALSE

GROUP BY pr.strategy;
"""

In [17]:
monthly_costs_sql = """
CREATE OR REPLACE VIEW
vw_monthly_costs AS

SELECT
    DATE_TRUNC(
        'month',
        pr.timestamp_utc
        AT TIME ZONE 'Europe/Madrid'
    ) AS month,

    pr.strategy,

    SUM(
        pr.incremental_cost_eur
    ) AS incremental_cost_eur,

    SUM(
        df.actual_demand_mw
    ) AS supplied_mwh,

    SUM(
        pr.incremental_cost_eur
    )
    /
    SUM(
        df.actual_demand_mw
    ) AS incremental_cost_eur_per_mwh

FROM procurement_results pr

JOIN demand_forecasts df
    ON pr.timestamp_utc
     = df.timestamp_utc

WHERE df.is_blackout = FALSE

GROUP BY
    DATE_TRUNC(
        'month',
        pr.timestamp_utc
        AT TIME ZONE 'Europe/Madrid'
    ),
    pr.strategy;
"""

In [18]:
monthly_savings_sql = """
CREATE OR REPLACE VIEW
vw_monthly_savings AS

SELECT
    month,

    MAX(
        CASE
            WHEN strategy = 'lag168'
            THEN incremental_cost_eur_per_mwh
        END
    ) AS lag168_cost_eur_per_mwh,

    MAX(
        CASE
            WHEN strategy = 'ml'
            THEN incremental_cost_eur_per_mwh
        END
    ) AS ml_cost_eur_per_mwh,

    MAX(
        CASE
            WHEN strategy = 'ree'
            THEN incremental_cost_eur_per_mwh
        END
    ) AS ree_cost_eur_per_mwh,

    MAX(
        CASE
            WHEN strategy = 'lag168'
            THEN incremental_cost_eur_per_mwh
        END
    )
    -
    MAX(
        CASE
            WHEN strategy = 'ml'
            THEN incremental_cost_eur_per_mwh
        END
    )
    AS ml_savings_vs_lag168_eur_per_mwh,

    MAX(
        CASE
            WHEN strategy = 'ml'
            THEN incremental_cost_eur_per_mwh
        END
    )
    -
    MAX(
        CASE
            WHEN strategy = 'ree'
            THEN incremental_cost_eur_per_mwh
        END
    )
    AS ree_savings_vs_ml_eur_per_mwh

FROM vw_monthly_costs

GROUP BY month;
"""

In [19]:
hourly_operations_sql = """
CREATE OR REPLACE VIEW
vw_hourly_operations AS

SELECT
    pr.timestamp_utc,
    pr.strategy,

    df.actual_demand_mw,

    pr.forecast_mw,
    pr.forecast_error_mw,
    pr.shortfall_mwh,
    pr.surplus_mwh,

    pr.incremental_cost_eur,

    mp.day_ahead_eur_mwh,
    mp.imbalance_up_eur_mwh,
    mp.imbalance_down_eur_mwh,

    df.is_blackout

FROM procurement_results pr

JOIN demand_forecasts df
    ON pr.timestamp_utc
     = df.timestamp_utc

JOIN market_prices mp
    ON pr.timestamp_utc
     = mp.timestamp_utc;
"""

In [20]:
view_definitions = [
    strategy_summary_sql,
    monthly_costs_sql,
    monthly_savings_sql,
    hourly_operations_sql,
]

with engine.begin() as conn:
    for sql in view_definitions:
        conn.execute(
            text(sql)
        )

## 6. Business queries

The analytical layer can now answer the key economic questions directly
in SQL.

In [21]:
strategy_summary = pd.read_sql(
    """
    SELECT *
    FROM vw_strategy_summary
    ORDER BY
        incremental_cost_eur_per_mwh;
    """,
    engine,
)

strategy_summary

,strategy,n_hours,incremental_cost_eur,supplied_mwh,incremental_cost_eur_per_mwh
0,ree,14572,1.640129e+08,4.118020e+08,0.398281
1,ml,14572,2.249585e+08,4.118020e+08,0.546278
2,lag168,14572,3.624379e+08,4.118020e+08,0.880127


In [22]:
assert set(
    strategy_summary["strategy"]
) == {
    "lag168",
    "ml",
    "ree",
}

assert (
    strategy_summary["n_hours"]
    == 14572
).all()

In [23]:
monthly_costs = pd.read_sql(
    """
    SELECT *
    FROM vw_monthly_costs
    ORDER BY
        month,
        strategy;
    """,
    engine,
)

monthly_costs.head(12)

,month,strategy,incremental_cost_eur,supplied_mwh,incremental_cost_eur_per_mwh
0,2025-01-01,lag168,3.638232e+06,2.155813e+07,0.168764
1,2025-01-01,ml,1.092652e+07,2.155813e+07,0.506840
2,2025-01-01,ree,1.396654e+07,2.155813e+07,0.647855
3,2025-02-01,lag168,2.405445e+07,1.906394e+07,1.261777
4,2025-02-01,ml,1.068487e+07,1.906394e+07,0.560475
5,2025-02-01,ree,1.310089e+07,1.906394e+07,0.687208
6,2025-03-01,lag168,2.406439e+07,2.046742e+07,1.175742
7,2025-03-01,ml,1.171201e+07,2.046742e+07,0.572227
8,2025-03-01,ree,1.548314e+07,2.046742e+07,0.756477
9,2025-04-01,lag168,1.952810e+07,1.716224e+07,1.137853


In [24]:
assert (
    monthly_costs["month"].min()
    == pd.Timestamp("2025-01-01")
)

In [25]:
monthly_savings = pd.read_sql(
    """
    SELECT *
    FROM vw_monthly_savings
    ORDER BY month;
    """,
    engine,
)

monthly_savings.head()

,month,lag168_cost_eur_per_mwh,ml_cost_eur_per_mwh,ree_cost_eur_per_mwh,ml_savings_vs_lag168_eur_per_mwh,ree_savings_vs_ml_eur_per_mwh
0,2025-01-01,0.168764,0.506840,0.647855,-0.338076,-0.141015
1,2025-02-01,1.261777,0.560475,0.687208,0.701302,-0.126732
2,2025-03-01,1.175742,0.572227,0.756477,0.603515,-0.184250
3,2025-04-01,1.137853,0.629686,0.688703,0.508167,-0.059018
4,2025-05-01,1.459755,0.557938,0.251733,0.901817,0.306204


In [26]:
best_ml_months = pd.read_sql(
    """
    SELECT
        month,
        ml_savings_vs_lag168_eur_per_mwh,
        ree_savings_vs_ml_eur_per_mwh

    FROM vw_monthly_savings

    ORDER BY
        ml_savings_vs_lag168_eur_per_mwh
        DESC

    LIMIT 5;
    """,
    engine,
)

best_ml_months

,month,ml_savings_vs_lag168_eur_per_mwh,ree_savings_vs_ml_eur_per_mwh
0,2025-09-01,1.180044,0.217291
1,2025-05-01,0.901817,0.306204
2,2025-07-01,0.722460,0.220319
3,2025-02-01,0.701302,-0.126732
4,2025-03-01,0.603515,-0.184250


## Conclusions

The forecasting pipeline is represented in PostgreSQL through three
normalized base tables: hourly demand forecasts, market prices and
strategy-level procurement results.

Four analytical views provide the reporting layer for strategy
performance, monthly procurement costs, monthly economic savings and
hourly operational analysis.

Blackout observations remain available in the base data, while the main
business views use normal-operation hours. Calendar-month aggregation is
performed in Europe/Madrid local time so that SQL reporting follows the
same temporal convention as the forecasting analysis.

The PostgreSQL database provides the persistence and reporting layer used
for downstream business intelligence analysis.